# Custom Algebras and Advanced Patterns

**Part II · Geometric Algebra** — Tutorial 16

This tutorial goes beyond the eight built-in basis classes. You can construct
geometric algebras with any dimension and signature (up to 31 dimensions), name
blades manually, build ad-hoc basis classes, and plug custom algebras into the
**numerical core** — `BladeMask`, the solver, matrix, tensor, and expression.

By the end you will be able to:

- Construct `Algebra(dim, sig, dtype)` for dimensions beyond the basis classes.
- Express a signature as a bitmask or a tuple of 1-based indices.
- Read and assign blade names with `blade_name` / `blade_id`.
- Use `BladeMask` and `solve` on a custom algebra.
- Define an ad-hoc basis class with named blades.
- Know when a custom algebra compiles on first use (needs a C++ toolchain).
- Understand why custom algebras cannot use the `Geometry` submodule.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (algebra construction),
> [Tutorial 10](../10_blade_mask/) (blade masks), and
> [Tutorial 11](../11_equation_solving/) (the solver).

## 1. Setup

Only the generic `Algebra` class is needed. The generic constructor builds custom
algebras; the built-in basis classes (`BasisE3`, …) pre-define named blades for
the standard models and are required by the `Geometry` submodule.

In [1]:
from pytanga import Algebra, BladeMask
from pytanga.solver.solve import solve

## 2. Precompiled bindings vs. compilation

The generic constructor can request **any** `(dim, sig)`. Five float64
signatures ship **precompiled** and load instantly with no toolchain: `(2,0)`,
`(3,0)`, `(4,0)`, `(4,8)`, `(5,16)` (plus `(3,0)` int64 for modular E3).
`Algebra(4, 0)` and `Algebra(4, 8)` below match those — `(4,0)` is the P3
signature and `(4,8)` is the N2/PGA2 signature. Any other combination — such as
`Algebra(5, 0)` below — is **compiled on first use** (~5–20 s) and therefore
needs a C++ toolchain (CMake + a compiler). See the
[README](../../../README.md) for setup.

## 3. Custom Euclidean algebras

`Algebra(dim, sig, dtype)` builds an algebra of arbitrary dimension. `G(4,0)` has
`2^4 = 16` blades and `G(5,0)` has `2^5 = 32` — neither is one of the eight
built-in basis classes.

In [2]:
G4 = Algebra(4, 0)      # 4D Euclidean — precompiled (P3 signature)
G5 = Algebra(5, 0)      # 5D Euclidean — compiled on first use

print("G(4,0):", G4)
print("  algebra_dim :", G4.algebra_dim)
print("G(5,0):", G5)
print("  algebra_dim :", G5.algebra_dim)

G(4,0): Algebra(dim=4, sig=0b00000000, dtype='float64')
  algebra_dim : 16
G(5,0): Algebra(dim=5, sig=0b00000000, dtype='float64')
  algebra_dim : 32


## 4. Custom signatures

The signature is a bitmask: bit `k` set means basis vector `e_{k+1}` squares to
`-1`. It may also be passed as a tuple of 1-based indices. Here `sig = 8` (bit 3)
makes `e4` square to `-1`.

In [3]:
euc = Algebra(4, 0)        # all basis vectors square to +1
sig = Algebra(4, 8)        # bitmask 0b1000: e4^2 = -1  (precompiled N2/PGA2 signature)
sig2 = Algebra(4, (4,))    # same signature via 1-based tuple

print("e4^2 (Euclidean):", (euc("e4") * euc("e4")).to_dict())
print("e4^2 (sig=8)    :", (sig("e4") * sig("e4")).to_dict())
print("same signature  :", sig.sig == sig2.sig, "->", sig.sig)

e4^2 (Euclidean): {'s': 1.0}
e4^2 (sig=8)    : {'s': -1.0}
same signature  : True -> 8


## 5. Custom algebras and the Geometry submodule

The `Geometry` submodule only accepts the built-in `BasisXX` classes. A
`Geometry` bound to a generic `Algebra` constructs, but `create` raises
`ValueError: Unknown basis type: Algebra` (and `analyze` returns `None`).
Custom algebras are therefore limited to the **numerical core** — `MV`,
`BladeMask`, the solver, matrix, tensor, and expression.

In [4]:
from pytanga.geometry import Geometry, Point

geo_sig = Geometry(sig)          # construction succeeds
try:
    geo_sig.create(Point(1, 2, 3))
except ValueError as err:
    print("create(Point) on a generic Algebra ->", err)

create(Point) on a generic Algebra -> Unknown basis type: Algebra


## 6. Blade names and ids

`blade_name(bid)` maps a blade bitmask to a name; `blade_id(name)` is the inverse.
The pseudoscalar is always named `I`.

In [5]:
print("blade_name(8)   :", G4.blade_name(8))
print("blade_name(15)  :", G4.blade_name(15))
print("blade_id('e14') :", G4.blade_id("e14"))
print("pseudoscalar_id :", G4.pseudoscalar_id, "->", G4.blade_name(G4.pseudoscalar_id))
print("all_blades      :", len(list(G4.all_blades())), "blades")

blade_name(8)   : e4
blade_name(15)  : I
blade_id('e14') : 9
pseudoscalar_id : 15 -> I
all_blades      : 16 blades


## 7. `BladeMask` on a custom algebra

A `BladeMask` binds to whichever algebra it is built from, so grade filters and
string expressions work unchanged on a 4D algebra.

In [6]:
vecs = BladeMask(G4, grades=[1])
bivs = BladeMask(G4, grades=[2])

print("vectors   :", vecs.ids)
print("bivectors :", bivs.names())

vectors   : [1, 2, 4, 8]
bivectors : ['e12', 'e13', 'e23', 'e14', 'e24', 'e34']


## 8. The solver on a custom algebra

The `solve` pipeline derives blade masks automatically, so it works on any
float algebra without change — here the multiplicative inverse of a vector in
`G(5,0)`.

In [7]:
A = G5("e1 + 2 e2 - e3")

X = solve(A, 1)
X.show("X = solve(A, 1)")
(A * X).prune().show("A * X")

X = solve(A, 1): 0.1667 e1 + 0.3333 e2 - 0.1667 e3

A * X: 1

## 9. Ad-hoc basis classes

Subclass `Algebra` to attach named blade attributes — the same pattern the
built-in `BasisE3` uses. This gives a reusable, named-blade algebra for a custom
model.

In [8]:
class G4Basis(Algebra):
    """A 4D Euclidean algebra with named basis blades."""
    def __init__(self, dtype="float64", **kw):
        super().__init__(4, 0, dtype, **kw)
        mv = self.multivector
        self.e1 = mv({1: 1})
        self.e2 = mv({2: 1})
        self.e3 = mv({4: 1})
        self.e4 = mv({8: 1})
        self.I = mv({15: 1})


B = G4Basis()
print("dim        :", B.dim)
print("e4         :", B.e4.to_dict())
print("I          :", B.I.to_dict())
print("e1 * e4    :", (B.e1 * B.e4).to_dict())

dim        : 4
e4         : {'e4': 1.0}
I          : {'I': 1.0}
e1 * e4    : {'e14': 1.0}


## 10. The compile-time cache

The C++ binding for each `(dim, sig, dtype)` combination is generated and compiled
once, then cached on disk. The first construction of a new combination may take a
few seconds; subsequent constructions load the cached binary in milliseconds. The
cache key is invalidated whenever the bundled C++ headers change, so a fresh
checkout rebuilds automatically.

The same engine powers `BladeMask`, `product_matrix`, and `solve` for any cached
algebra — including non-Euclidean signatures such as `Algebra(4, 8)` above. The
precompiled wheel ships the standard signatures, so the common cases never
compile at all.

## 11. Summary & next steps

| Task | API |
|---|---|
| Custom Euclidean algebra | `Algebra(4, 0)`, `Algebra(5, 0)` |
| Custom signature | `Algebra(4, 8)` or `Algebra(4, (4,))` |
| Blade name / id | `alg.blade_name(bid)`, `alg.blade_id(name)` |
| Blade enumeration | `alg.all_blades()`, `alg.pseudoscalar_id` |
| Mask / solver on custom algebra | `BladeMask(alg, …)`, `solve(A, 1)` |
| Ad-hoc basis class | subclass `Algebra` and add named blades |
| Precompiled signatures | `(2,0)`, `(3,0)`, `(4,0)`, `(4,8)`, `(5,16)` |
| Geometry on a custom algebra | not supported — `create` raises `ValueError` |

**Where to go next:**

- [**17 · Visualizing Algebra Entities**](../17_visualizing_algebra_entities/) —
  the closing chapter that pictures everything built so far.